<div id="singlestore-header" style="display: flex; background-color: rgba(124, 195, 235, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/arrows-spin.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Operationalize Integrations with Flow and Connectors</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook requires a Standard or Premium workspace; it does not run on a Free Starter Workspace. The SingleStore Flow APIs are currently a <b>Preview</b> feature. The notebook runs end to end in dry-run mode without a Flow instance.</p>
    </div>
</div>

Demonstrate console setup, Flow APIs, automation, CDC, and commonly used source connectors.

### What this notebook demonstrates

| # | Section | What the audience sees |
|---|---------|------------------------|
| 0 | **Getting started** | Install packages, load secrets (Flow endpoint, API key, DB credentials), and choose dry-run or live mode |
| 1 | **Console setup** | How a Flow on Helios instance is provisioned from the Cloud Portal, and what the checklist looks like |
| 2 | **Source connectors** | The commonly used sources (MySQL, PostgreSQL, SQL Server, Oracle, Snowflake, MongoDB, any JDBC) and how each one captures change |
| 3 | **Flow APIs** | A small Python client for the Ingest and XL Ingest REST APIs |
| 4 | **Automation** | One idempotent function that stands up a full pipeline: connections → tables → schedule → full extract |
| 5 | **CDC** | Make changes on the source, trigger/await a delta, and prove they landed in SingleStore |
| 6 | **Large tables** | Parallel, sliced initial load with XL Ingest |
| 7 | **Operate it** | Schedules, schema sync, freshness checks, and where native Pipelines (Kafka, S3) fit |

> **Run modes.** Every API call goes through one client with a `DRY_RUN` switch. With `DRY_RUN = True` (the default) the notebook prints the exact HTTP request it *would* send, so you can rehearse the whole demo with no Flow instance. Flip it to `False` once your Flow endpoint and API key are set.

> **Status note.** The Flow APIs are documented by SingleStore as a **Preview** feature. Endpoint shapes below follow the current Flow on Helios API reference; check the docs before a customer-facing run.

### Architecture at a glance

```
 ┌──────────────── Sources ────────────────┐        ┌────────── SingleStore Flow (on Helios) ──────────┐        ┌──── SingleStore Helios ────┐
 │ MySQL ........ binlog (Log / Continuous)│        │  Ingest      schema + data ≤10 GB + ongoing CDC  │        │                            │
 │ PostgreSQL ... logical replication (WAL)│ ─────▶ │  XL Ingest   parallel sliced load for big tables │ ─────▶ │  Standard workspace / DB   │
 │ SQL Server ... Change Tracking or CDC   │        │                                                  │        │  (tables w/ PK + shard key │
 │ Oracle ....... LogMiner                 │        │  Control plane:  Portal UI  ·  REST API (Bearer) │        │   created automatically)   │
 │ Snowflake / any JDBC ... full extracts  │        └──────────────────────────────────────────────────┘        └────────────────────────────┘
 │ MongoDB ...... private preview          │                    ▲
 └─────────────────────────────────────────┘                    │  this notebook: FlowClient → provision → CDC check → monitor
```

**Native Pipelines** (Kafka, S3, GCS, Azure Blob, …) are the complementary path for *streams and files*; Flow is the path for *operational databases with CDC*. Section 7 shows both side by side.

---
## 0 · Getting started

Store credentials as **Secrets** in the SingleStore Portal (*Notebooks → Secrets*) rather than in cells. The helper below reads a Portal secret when running inside a SingleStore notebook, falls back to environment variables anywhere else, and finally to the placeholder default.

| Secret name | Used for |
|---|---|
| `FLOW_API_KEY` | Bearer token for the Flow API (Portal → *Configuration → API keys*) |
| `FLOW_ENDPOINT` | Flow instance endpoint (Portal → *Ingestion → Flow* → ⋯ → *View Details* → **Endpoint**) |
| `SRC_HOST`, `SRC_PORT`, `SRC_USER`, `SRC_PASSWORD`, `SRC_DATABASE` | Source database |
| `S2_HOST`, `S2_USER`, `S2_PASSWORD` | Destination SingleStore workspace |

In [1]:
# Optional: only needed for the live CDC section (writing to a MySQL source) and destination checks
%pip install --quiet requests pandas singlestoredb pymysql

In [2]:
import os, json, time, copy, random, string, urllib.parse
from datetime import datetime, timezone

import pandas as pd
import requests

try:
    from IPython.display import display, Markdown
except ImportError:            # running outside Jupyter
    display = print
    Markdown = str

def secret(name, default=None):
    """Portal secret -> environment variable -> default."""
    try:
        return get_secret(name)          # provided automatically inside SingleStore notebooks
    except Exception:
        return os.environ.get(name, default)

# ---------------------------------------------------------------- demo switches
DRY_RUN = True          # True = print requests only. Set False to call your live Flow instance.
SOURCE_KIND = "mysql"   # mysql | postgres | sqlserver | oracle | snowflake  (drives the source connector config)

# ---------------------------------------------------------------- Flow on Helios endpoint
# Format: https://gateway.<cell>.svc.singlestore.com/ingest/<flow-id>
#  - Ingest API base   = <endpoint>/ingest/api/ingest
#  - XL Ingest API base = same gateway, /xlingest/<flow-id>/ixl
FLOW_ENDPOINT = secret("FLOW_ENDPOINT", "https://gateway.aws-virginia-8.svc.singlestore.com/ingest/00000000-0000-0000-0000-000000000000")
FLOW_API_KEY  = secret("FLOW_API_KEY",  "<your-api-key>")

# ---------------------------------------------------------------- source (example: MySQL classicmodels)
SRC = dict(
    host     = secret("SRC_HOST", "mysql.example.com"),
    port     = secret("SRC_PORT", "3306"),
    user     = secret("SRC_USER", "bflow_ingest_user"),
    password = secret("SRC_PASSWORD", "<source-password>"),
    database = secret("SRC_DATABASE", "classicmodels"),
)

# ---------------------------------------------------------------- destination (SingleStore Helios)
S2 = dict(
    host     = secret("S2_HOST", "svc-xxxxxxxx-dml.aws-virginia-8.svc.singlestore.com"),
    port     = secret("S2_PORT", "3306"),
    user     = secret("S2_USER", "admin"),
    password = secret("S2_PASSWORD", "<s2-password>"),
    database = secret("S2_DATABASE", "classicmodels"),
)

DEMO_TABLES = ["customers", "orders", "orderdetails", "products"]

print(f"DRY_RUN={DRY_RUN} | source={SOURCE_KIND} {SRC['host']}:{SRC['port']}/{SRC['database']} -> SingleStore {S2['host']}")

---
## 1 · Console setup (Flow on Helios)

Flow on Helios is fully managed: there is nothing to install, and the instance is billed only while it runs. The console path is the fastest way to show "zero to replicating" live.

**Walkthrough to show on screen**

1. **Cloud Portal → Ingestion → Load Data.**
2. **Pick a source tile**: *MySQL by Flow*, *PostgreSQL by Flow*, *Oracle by Flow*, *SQL Server by Flow*, or *Snowflake by Flow*.
3. **Select destination**: choose the workspace (must be the **Standard** type) and database; create or reuse a named connection (no leading digit, no special characters except `_`).
4. **Open Dashboard** → choose **Ingest** (≤ 10 GB per table, plus ongoing CDC) or **XL Ingest** (larger tables) from the top-right drop-down.
5. **Complete the Checklist**: *Setup Source → Setup Destination → Setup Tables → Setup Schedule*.
6. **Operations tab → start the pipeline.** It then runs on the chosen schedule.
7. **Monitor**: *Operations* (re-sync, schema changes), *Settings*, *Reports* (per-table performance by day/month), *Logs* (extract/load progress).
8. **Lifecycle**: suspend/resume the instance to stop compute; scale it up or down as volume changes.

**Before the demo — network**
- Find the **outbound IPs** of your Helios workspace and add them to the source's firewall / security group / allowed-IP list.
- Self-hosted or on-prem sources must be reachable from the internet for Flow on Helios. If the source sits deep on-prem with high volume, the **self-managed Flow** deployment (runs next to the source) is the better fit.

**Then grab two values for the API sections:** the instance **Endpoint** (*Ingestion → Flow → ⋯ → View Details*) and an **API key** (*Configuration → API keys → Create API key*; it's shown only once).

> 💡 **Talk track:** "Everything you just clicked is also an API call. The rest of this notebook does the same thing from code, so it can go into CI/CD, Terraform wrappers, or an onboarding script that spins up a pipeline per tenant."

---
## 2 · Commonly used source connectors

Flow's connector `type` decides *how* change is captured. Picking the right one is the main design decision per source.

In [3]:
CONNECTORS = pd.DataFrame([
    # source,                     type code,            capture mechanism,            CDC?,  default port, notes
    ("MySQL 5.1+",                "rds.mysqlc",         "Binlog – Continuous",        "Yes", 3306, "Real-time CDC; recommended for MySQL / RDS / Aurora"),
    ("MySQL 5.1+",                "rds.mysql",          "Binlog – Log (polled)",      "Yes", 3306, "CDC on the configured schedule"),
    ("PostgreSQL 9.4+",           "rds.pgres",          "Logical replication (WAL)",  "Yes", 5432, "Needs wal_level=logical + replication slot"),
    ("SQL Server",                "rds.mscdcm",         "SQL Server CDC",             "Yes", 1433, "Requires SQL Server Agent running"),
    ("SQL Server",                "rds.msadbctm",       "Change Tracking",            "Yes", 1433, "Lighter weight than CDC"),
    ("SQL Server",                "rds.msfull",         "Full extracts",              "No",  1433, "Snapshot only"),
    ("Oracle 19c",                "rds.oracle11lmc19",  "Continuous LogMiner",        "Yes", 1521, "ARCHIVELOG + supplemental logging"),
    ("Oracle",                    "rds.oracle11lfm",    "Fast LogMiner",              "Yes", 1521, "Also RAC / PDB variants"),
    ("Oracle",                    "rds.oraclets",       "Timestamps",                 "Yes*",1521, "*Incremental by timestamp column"),
    ("MongoDB / Atlas",           "rds.mongo / rds.mongoatlas", "Change streams",      "Yes", 27017,"Private preview"),
    ("Snowflake / any JDBC",      "rds.jdbcfe",         "Full extracts via JDBC",     "No",  None, "Snowflake JDBC is GA; other JDBC sources are preview"),
], columns=["Source", "Flow type", "Capture mechanism", "CDC", "Port", "Notes"])

display(CONNECTORS)

# The connector picked for this run
SOURCE_TYPE = {
    "mysql":     "rds.mysqlc",
    "postgres":  "rds.pgres",
    "sqlserver": "rds.mscdcm",
    "oracle":    "rds.oracle11lmc19",
    "snowflake": "rds.jdbcfe",
}[SOURCE_KIND]
print("Using Flow source type:", SOURCE_TYPE)

### 2.1 · Preparing each source for CDC

Show the one that matches your audience; the others are reference. Run these on the **source**, not in SingleStore.

In [4]:
SOURCE_PREP = {
"mysql": """
-- my.cnf / RDS or Aurora parameter group, then restart
--   server_id=1   log_bin=/var/lib/mysql/binlog   binlog_format=ROW
--   binlog_row_image=FULL   binlog_checksum=NONE (or CRC32)
--   binlog_expire_logs_seconds=604800   -- 7 days; MySQL 8.2+
CREATE USER 'bflow_ingest_user' IDENTIFIED BY '<password>';
GRANT SELECT, REPLICATION CLIENT, REPLICATION SLAVE, SHOW DATABASES, PROCESS ON *.* TO 'bflow_ingest_user';
-- verify
SHOW VARIABLES WHERE Variable_name IN ('log_bin','binlog_format','binlog_row_image','binlog_checksum');
""",
"postgres": """
-- postgresql.conf, then restart
--   wal_level = logical   max_replication_slots >= tasks   max_wal_senders > 1   wal_sender_timeout = 0
-- pg_hba.conf:  host  replication  all  <flow-outbound-ip>/32  md5
SELECT pg_create_logical_replication_slot('bflow_replication_slot', 'test_decoding');
-- Flow user needs replication role (+ superuser per current docs) and read access to replicated tables
-- Remember: drop unused slots, or WAL accumulates
""",
"sqlserver": """
-- Option A: SQL Server CDC  (type rds.mscdcm) -- SQL Server Agent must be running
USE <database_name>;
EXEC sys.sp_cdc_enable_db;
EXEC sys.sp_cdc_enable_table @source_schema = N'dbo', @source_name = N'<table>', @role_name = NULL, @supports_net_changes = 0;
EXEC sys.sp_cdc_change_job @job_type = N'cleanup', @retention = 10080;   -- minutes (default is 3 days)

-- Option B: Change Tracking  (type rds.msadbctm)
ALTER DATABASE <database_name> SET CHANGE_TRACKING = ON (CHANGE_RETENTION = 7 DAYS, AUTO_CLEANUP = ON);
ALTER TABLE <table> ENABLE CHANGE_TRACKING WITH (TRACK_COLUMNS_UPDATED = ON);
GRANT VIEW SERVER STATE TO <login>;
GRANT VIEW CHANGE TRACKING ON <schema>.<table> TO <user>;
""",
"oracle": """
-- Must be in ARCHIVELOG mode:  SELECT log_mode FROM V$DATABASE;
ALTER DATABASE ADD SUPPLEMENTAL LOG DATA (ALL) COLUMNS;
ALTER TABLE <schema>.<table> ADD SUPPLEMENTAL LOG DATA (ALL) COLUMNS;
-- (Amazon RDS: exec rdsadmin.rdsadmin_util.alter_supplemental_logging('ADD','ALL');
--              exec rdsadmin.rdsadmin_util.set_configuration('archivelog retention hours', 24);)
GRANT CREATE SESSION, LOGMINING, SELECT ANY TRANSACTION, EXECUTE_CATALOG_ROLE TO <user>;
GRANT SELECT ON V_$ARCHIVED_LOG TO <user>;   GRANT SELECT ON V_$LOGMNR_CONTENTS TO <user>;
GRANT SELECT ON V_$LOGMNR_LOGS  TO <user>;   GRANT SELECT ON DBA_OBJECTS        TO <user>;
GRANT EXECUTE ON DBMS_LOGMNR    TO <user>;
-- 19c continuous mining also: V_$DATABASE, V_$LOG, V_$LOGFILE
""",
"snowflake": """
-- Full extracts only (no CDC). Flow user needs SELECT on each table:
GRANT SELECT ON TABLE <db>.<schema>.<table> TO ROLE <flow_role>;
""",
}
print(SOURCE_PREP[SOURCE_KIND])

---
## 3 · The Flow APIs

Flow exposes two REST APIs, both authenticated with `Authorization: Bearer <API key>`:

| API | Base URL (Flow on Helios) | Purpose |
|---|---|---|
| **Ingest** | `https://gateway.<cell>.svc.singlestore.com/ingest/<flow-id>/ingest/api/ingest` | Connections, schedule, table selection, operations (full / delta / syncstruct / syncnew) |
| **XL Ingest** | `https://gateway.<cell>.svc.singlestore.com/xlingest/<flow-id>/ixl` | Slice large tables and load slices in parallel |

Two conventions worth pointing out to the audience:
- Connection payloads are **arrays of `{"name": ..., "value": ...}` pairs**, not plain objects.
- *Save* endpoints (`PATCH`) return **204** and do not validate; always follow with the matching `/test` call.

The self-managed Flow app exposes the same paths at `http://<host>:<port>/ingest/api/ingest` and `/ixl` (no Bearer token). The client below supports both.

In [5]:
class FlowError(RuntimeError):
    pass


class FlowClient:
    """Thin client for the SingleStore Flow Ingest + XL Ingest APIs (Preview)."""

    SECRET_KEYS = {"pwd", "pwd2"}

    def __init__(self, endpoint, api_key=None, dry_run=True, timeout=60, verbose=True):
        self.endpoint = endpoint.rstrip("/")
        self.ingest_base = f"{self.endpoint}/ingest/api/ingest"
        # Helios: .../ingest/<flow-id>  ->  .../xlingest/<flow-id>/ixl ; self-managed: <host>/ixl
        if "/ingest/" in self.endpoint:
            gw, flow_id = self.endpoint.rsplit("/ingest/", 1)
            self.xl_base = f"{gw}/xlingest/{flow_id}/ixl"
        else:
            self.xl_base = f"{self.endpoint}/ixl"
        self.api_key, self.dry_run, self.timeout, self.verbose = api_key, dry_run, timeout, verbose
        self.session = requests.Session()
        self.history = []                      # audit trail of every call (great for the demo)

    # ------------------------------------------------------------------ plumbing
    def _headers(self):
        h = {"Content-Type": "application/json", "Accept": "application/json"}
        if self.api_key:
            h["Authorization"] = f"Bearer {self.api_key}"
        return h

    @classmethod
    def _mask(cls, body):
        if isinstance(body, list):
            return [{**p, "value": "***"} if p.get("name") in cls.SECRET_KEYS else p for p in body]
        return body

    def _call(self, method, url, body=None, dry_result=None):
        entry = {"ts": datetime.now(timezone.utc).strftime("%H:%M:%S"), "method": method,
                 "path": ("[XL] " + url.split("/ixl", 1)[1]) if "/ixl" in url
                         else (url.replace(self.ingest_base, "") or "/"), "status": None}
        if self.verbose:
            print(f"→ {method} {url}")
            if body is not None:
                print("  " + json.dumps(self._mask(body)))
        if self.dry_run:
            entry["status"] = "dry-run"
            self.history.append(entry)
            return copy.deepcopy(dry_result) if dry_result is not None else {"success": True, "dry_run": True}

        r = self.session.request(method, url, headers=self._headers(),
                                 data=None if body is None else json.dumps(body), timeout=self.timeout)
        entry["status"] = r.status_code
        self.history.append(entry)
        if r.status_code >= 400:
            raise FlowError(f"{method} {url} -> HTTP {r.status_code}: {r.text[:500]}")
        if r.status_code == 204 or not r.content:
            return {"success": True}
        try:
            data = r.json()
        except ValueError:
            return {"success": True, "raw": r.text}
        # Flow reports many failures as HTTP 200 + success:false
        if isinstance(data, dict) and data.get("success") is False:
            msgs = "; ".join(m.get("message", "") for m in data.get("messageList", [])) or data.get("message", "")
            raise FlowError(f"{method} {url} -> {msgs or data}")
        return data

    @staticmethod
    def _pairs(cfg):
        return [{"name": k, "value": str(v)} for k, v in cfg.items() if v is not None]

    @staticmethod
    def connection(host, port, user, password, type_, db=None, drv=None, url=None):
        # pwd2 (confirmation) is still accepted by the API; harmless if the server ignores it
        return dict(host=host, port=port, uid=user, pwd=password, pwd2=password, type=type_, db=db, drv=drv, url=url)

    # ------------------------------------------------------------------ Ingest: connections
    def save_source(self, cfg):        return self._call("PATCH", f"{self.ingest_base}/conn-src", self._pairs(cfg))
    def test_source(self, cfg):        return self._call("POST",  f"{self.ingest_base}/conn-src/test", self._pairs(cfg))
    def get_source(self):              return self._call("GET",   f"{self.ingest_base}/conn-src",
                                                         dry_result=[{"name": "status", "value": "Y"}, {"name": "type", "value": SOURCE_TYPE}])
    def save_destination(self, cfg):   return self._call("PATCH", f"{self.ingest_base}/conn-dst", self._pairs(cfg))
    def test_destination(self, cfg):   return self._call("POST",  f"{self.ingest_base}/conn-dst/test", self._pairs(cfg))
    def get_destination(self):         return self._call("GET",   f"{self.ingest_base}/conn-dst",
                                                         dry_result=[{"name": "status", "value": "Y"}, {"name": "type", "value": "rds.sstorem"}])

    # ------------------------------------------------------------------ Ingest: schedule
    def set_schedule(self, type_="periodic", duration=None, offset=None, week_flags=None):
        body = {"type": type_}
        if duration:   body["duration"] = duration
        if offset:     body["offset"] = offset
        if week_flags: body["weekFlags"] = week_flags
        return self._call("PATCH", f"{self.ingest_base}/config-sched", body)

    def get_schedule(self):
        return self._call("GET", f"{self.ingest_base}/config-sched",
                          dry_result={"type": "periodic", "duration": "0d00h01m00s", "offset": "0d00h00m", "isScheduled": True, "weekFlags": None})

    # ------------------------------------------------------------------ Ingest: tables
    def list_tables(self, database, schema):
        fake = [{"name": t, "selected": False, "id": t} for t in
                ["customers", "employees", "offices", "orderdetails", "orders", "payments", "productlines", "products"]]
        return self._call("GET", f"{self.ingest_base}/list-db/{database}/{schema}", dry_result=fake)

    def select_table(self, database, schema, table, select=True):
        return self._call("PATCH", f"{self.ingest_base}/config-tab/{database}/{schema}/{table}",
                          {"select": "Y" if select else "N"})

    # ------------------------------------------------------------------ Ingest: operations
    def run(self, operation):
        assert operation in {"full", "delta", "syncstruct", "syncnew"}, operation
        labels = {"full": "Full Extract", "delta": "Delta", "syncstruct": "Synchronize Structure", "syncnew": "Sync New Tables"}
        return self._call("POST", f"{self.ingest_base}/ops/extract/{operation}",
                          dry_result={"success": True, "message": f"{labels[operation]} requested"})

    # ------------------------------------------------------------------ XL Ingest
    @staticmethod
    def xl_table_id(database, schema, table):
        return f"{database}:{schema}.{table}"

    def xl_configure(self, table_id, slice_col, slice_bounds=None, slice_count=None, prefix_size=None, where=""):
        """Manual slicing: pass slice_bounds. Automatic slicing: pass slice_count + prefix_size."""
        body = {"id": table_id, "select": True, "sliceCol": slice_col, "where": where,
                "sliceList": [{"name": str(b)} for b in (slice_bounds or [])],
                "prefixSize": "" if prefix_size is None else prefix_size,
                "sliceCount": "" if slice_count is None else slice_count}
        return self._call("PATCH", f"{self.xl_base}?cmd=tdp", body)

    def xl_transfer(self, table_id):
        tab = urllib.parse.quote(table_id, safe=".")
        n = 4
        return self._call("POST", f"{self.xl_base}?cmd=tgsy&tab={tab}",
                          dry_result={"isRunning": True, "type": "SY", "tasksComplete": 0, "table": table_id, "tasksTotal": n})

    # ------------------------------------------------------------------ demo helpers
    def audit(self):
        return pd.DataFrame(self.history)


flow = FlowClient(FLOW_ENDPOINT, FLOW_API_KEY, dry_run=DRY_RUN)
print("Ingest API   :", flow.ingest_base)
print("XL Ingest API:", flow.xl_base)

### 3.1 · Connections — save, then test

Flow keeps one source and one destination per instance. The destination type is always `rds.sstorem` (SingleStore).

In [6]:
src_cfg = FlowClient.connection(SRC["host"], SRC["port"], SRC["user"], SRC["password"], SOURCE_TYPE, db=SRC["database"])
dst_cfg = FlowClient.connection(S2["host"],  S2["port"],  S2["user"],  S2["password"],  "rds.sstorem", db=S2["database"])

flow.save_source(src_cfg)
print("source test:", flow.test_source(src_cfg), "\n")

flow.save_destination(dst_cfg)
print("destination test:", flow.test_destination(dst_cfg))

In [7]:
# Read back what Flow stored. Passwords come back masked; status "Y" = last test succeeded.
def pairs_to_dict(pairs):
    return {p["name"]: p["value"] for p in pairs} if isinstance(pairs, list) else pairs

display(pd.DataFrame([pairs_to_dict(flow.get_source()), pairs_to_dict(flow.get_destination())], index=["source", "destination"]))

### 3.2 · Discover and select tables

`list-db/{database}/{schema}` returns every table plus whether it's selected. For MySQL the database and schema are typically the same name; for SQL Server/Postgres the schema is usually `dbo` / `public`.

In [8]:
SCHEMA = {"mysql": SRC["database"], "postgres": "public", "sqlserver": "dbo"}.get(SOURCE_KIND, SRC["database"])

tables = pd.DataFrame(flow.list_tables(SRC["database"], SCHEMA))
display(tables)

for t in DEMO_TABLES:
    flow.select_table(SRC["database"], SCHEMA, t)

### 3.3 · Schedule

| type | `duration` format | Example | Use when |
|---|---|---|---|
| `manual` | – | `{"type":"manual"}` | Demos, one-off migrations, orchestrator-triggered deltas |
| `auto` | – | `{"type":"auto"}` | Let Flow run continuously |
| `periodic` | `DdHHhMMmSSs` | `0d00h01m00s` | Near-real-time CDC every N minutes |
| `daily` | `HHhMMm` | `02h30m` | Nightly batch |
| `weekly` | `HHhMMm` + `weekFlags` (Sun→Sat) | `NYYYYYN` | Weekday-only loads |

In [9]:
flow.set_schedule("periodic", duration="0d00h01m00s", offset="0d00h00m")   # every minute for the demo
display(flow.get_schedule())

# Other patterns (uncomment to show):
# flow.set_schedule("daily",  duration="02h30m")
# flow.set_schedule("weekly", duration="06h00m", week_flags="NYYYYYN")
# flow.set_schedule("manual")

### 3.4 · Initial load

`POST /ops/extract/full` creates the schema in SingleStore (primary and shard keys included) and copies every selected table. The call returns immediately; progress shows in the dashboard **Logs** tab — keep it open on a second screen.

In [10]:
print(flow.run("full"))

---
## 4 · Automation: one call to stand up a pipeline

The cells above, packaged as an **idempotent** function you can drop into a CI job, an Airflow/Dagster task, or a tenant-onboarding script. Re-running it is safe: saves are upserts and table selection is declarative.

In [11]:
def provision_pipeline(flow, source_cfg, dest_cfg, database, schema, tables,
                       schedule=("periodic", "0d00h05m00s"), initial_load=True):
    """Configure + validate + start a Flow Ingest pipeline. Returns a summary dict."""
    steps = []
    def step(name, fn, *a, **kw):
        t0 = time.time()
        out = fn(*a, **kw)
        steps.append({"step": name, "seconds": round(time.time() - t0, 2), "result": str(out)[:80]})
        return out

    step("save source",        flow.save_source, source_cfg)
    step("test source",        flow.test_source, source_cfg)          # raises FlowError on failure
    step("save destination",   flow.save_destination, dest_cfg)
    step("test destination",   flow.test_destination, dest_cfg)

    available = {t["name"] for t in step("list tables", flow.list_tables, database, schema)}
    missing = set(tables) - available
    if missing:
        raise FlowError(f"Tables not found on source: {sorted(missing)}")
    for t in tables:
        step(f"select {t}", flow.select_table, database, schema, t)

    stype, dur = schedule
    step("set schedule", flow.set_schedule, stype, duration=dur)
    if initial_load:
        step("full extract", flow.run, "full")
    return pd.DataFrame(steps)


quiet = FlowClient(FLOW_ENDPOINT, FLOW_API_KEY, dry_run=DRY_RUN, verbose=False)
display(provision_pipeline(quiet, src_cfg, dst_cfg, SRC["database"], SCHEMA, DEMO_TABLES,
                           schedule=("periodic", "0d00h01m00s")))

**Patterns to mention**
- **GitOps:** keep `{source, tables, schedule}` per pipeline in YAML; a CI job calls `provision_pipeline` on merge.
- **Secrets:** API keys and DB passwords come from the Portal secret store / your vault — never from the repo.
- **Many sources:** one Flow instance = one source → one destination. Fan out by creating one instance per source and looping over their endpoints.
- **Audit:** every call is recorded; see below.

In [12]:
display(flow.audit().tail(15))

---
## 5 · CDC in action

The demo loop:
1. Snapshot row counts in SingleStore.
2. Make inserts, updates and a delete on the **source**.
3. Wait for the schedule (or force a `delta`).
4. Show the changes in SingleStore.

With `DRY_RUN = True` the source/destination cells are skipped and the narrative is printed instead.

In [13]:
import contextlib

@contextlib.contextmanager
def s2_conn():
    import singlestoredb as s2
    try:
        conn = s2.connect(connection_url)          # defined automatically inside SingleStore notebooks
    except NameError:
        conn = s2.connect(host=S2["host"], port=int(S2["port"]), user=S2["user"],
                          password=S2["password"], database=S2["database"])
    try:
        yield conn
    finally:
        conn.close()

def s2_query(sql, params=None):
    with s2_conn() as c, c.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description] if cur.description else []
        return pd.DataFrame(cur.fetchall(), columns=cols)

def dest_counts(tables=DEMO_TABLES):
    if DRY_RUN:
        return pd.DataFrame({"table": tables, "rows": ["(dry-run)"] * len(tables)})
    return pd.concat([s2_query(f"SELECT '{t}' AS `table`, COUNT(*) AS `rows` FROM `{S2['database']}`.`{t}`")
                      for t in tables], ignore_index=True)

before = dest_counts()
display(before)

In [14]:
# Generate change on the SOURCE (MySQL example). Uses a tagged customer so the demo is easy to find and clean up.
DEMO_TAG = "FLOWDEMO-" + "".join(random.choices(string.ascii_uppercase, k=4))
DEMO_CUSTOMER = 990000 + random.randint(0, 9999)

source_changes = [
    ("INSERT customers", f"""INSERT INTO customers (customerNumber, customerName, contactLastName, contactFirstName,
                                phone, addressLine1, city, country, creditLimit)
                             VALUES ({DEMO_CUSTOMER}, '{DEMO_TAG} Corp', 'Demo', 'Flow', '555-0100',
                                     '1 Pipeline Way', 'New York', 'USA', 50000.00)"""),
    ("UPDATE customers", f"UPDATE customers SET creditLimit = 75000.00 WHERE customerNumber = {DEMO_CUSTOMER}"),
    ("UPDATE products",  "UPDATE products SET quantityInStock = quantityInStock - 1 WHERE productCode = 'S10_1678'"),
]

if DRY_RUN or SOURCE_KIND != "mysql":
    for label, sql in source_changes:
        print(f"[would run on source] {label}:\n  {' '.join(sql.split())}\n")
else:
    import pymysql
    with pymysql.connect(host=SRC["host"], port=int(SRC["port"]), user=SRC["user"], password=SRC["password"],
                         database=SRC["database"], autocommit=True) as src:
        with src.cursor() as cur:
            for label, sql in source_changes:
                cur.execute(sql)
                print(f"✔ {label} ({cur.rowcount} row)")
change_made_at = datetime.now(timezone.utc)

In [15]:
# Don't want to wait for the schedule? Force a delta now.
print(flow.run("delta"))

In [16]:
def wait_for_change(timeout_s=180, poll_s=10):
    """Poll SingleStore until the demo row shows the updated value. Returns observed replication lag."""
    if DRY_RUN:
        print("(dry-run) would poll SingleStore for", DEMO_TAG)
        return None
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        df = s2_query(f"SELECT customerNumber, customerName, creditLimit FROM `{S2['database']}`.customers "
                      f"WHERE customerNumber = %s", (DEMO_CUSTOMER,))
        if len(df) and float(df.creditLimit.iloc[0]) == 75000.0:
            lag = (datetime.now(timezone.utc) - change_made_at).total_seconds()
            display(df)
            print(f"✔ change visible in SingleStore after ~{lag:.0f}s")
            return lag
        print("…waiting"); time.sleep(poll_s)
    raise TimeoutError("Change not visible yet — check the Logs tab in the Flow dashboard")

lag = wait_for_change()
display(dest_counts())

In [17]:
# Delete on the source -> delete in SingleStore (also cleans up the demo row)
cleanup_sql = f"DELETE FROM customers WHERE customerNumber = {DEMO_CUSTOMER}"
if DRY_RUN or SOURCE_KIND != "mysql":
    print("[would run on source]", cleanup_sql)
else:
    with pymysql.connect(host=SRC["host"], port=int(SRC["port"]), user=SRC["user"], password=SRC["password"],
                         database=SRC["database"], autocommit=True) as src, src.cursor() as cur:
        cur.execute(cleanup_sql); print("✔ deleted on source; it will disappear from SingleStore on the next delta")

### 5.1 · Schema changes

When the source table changes shape, `syncstruct` pushes the new structure to SingleStore. Recent Flow releases also detect column adds/drops during CDC for **MySQL and PostgreSQL** automatically (schema evolution), so `syncstruct` is mostly needed for other sources or for an explicit, controlled sync.

When someone adds a brand-new table on the source, select it and call `syncnew` to load just that table without re-running a full extract.

In [18]:
# e.g. after: ALTER TABLE customers ADD COLUMN loyaltyTier VARCHAR(16);
print(flow.run("syncstruct"))

# e.g. after creating a new table 'returns' on the source:
flow.select_table(SRC["database"], SCHEMA, "payments")
print(flow.run("syncnew"))

---
## 6 · Large tables with XL Ingest

Ingest handles tables up to ~10 GB. Beyond that, XL Ingest splits the initial copy into **slices** on a column and loads them in parallel. It is resilient to restarts of the network, Flow, source or destination — a failed slice is retried, not the whole table. Ongoing CDC is still done by Ingest.

**Order of operations**
1. In Ingest, select the big table and enable **Skip Initial Extract** for it (dashboard → *Tables*).
2. Run a full extract in Ingest — this copies only the schema for that table.
3. Configure slices in XL Ingest (`cmd=tdp`) and start the transfer (`cmd=tgsy`).

In [19]:
BIG_TABLE = flow.xl_table_id("default", SCHEMA, "orderdetails")   # format: <db>:<schema>.<table>

# Option A – manual slices: boundaries on a numeric/date key
flow.xl_configure(BIG_TABLE, slice_col="orderNumber",
                  slice_bounds=[10200, 10300, 10400])

# Option B – automatic slices (pick one option per table)
# flow.xl_configure(BIG_TABLE, slice_col="orderNumber", slice_count=8, prefix_size=2)

status = flow.xl_transfer(BIG_TABLE)
display(status)
if status.get("error"):
    print("⚠", status["error"])
else:
    print(f"{status['tasksComplete']}/{status['tasksTotal']} slices complete — progress is visible in the XL Ingest dashboard")

**Choosing a slice column:** an indexed, evenly distributed key (order id, created date). Aim for slices of a few GB each; use `where` to skip archived rows (e.g. `"created_at >= '2024-01-01'"`).

---
## 7 · Operating integrations

### 7.1 · Freshness and reconciliation from SingleStore

Flow's dashboard covers pipeline health. Applications usually also want a data-level check from the destination side. A simple reconciliation compares source and destination counts per table.

In [20]:
def reconcile(tables=DEMO_TABLES):
    if DRY_RUN or SOURCE_KIND != "mysql":
        return pd.DataFrame({"table": tables, "source_rows": "-", "dest_rows": "-", "match": "(dry-run)"})
    import pymysql
    rows = []
    with pymysql.connect(host=SRC["host"], port=int(SRC["port"]), user=SRC["user"], password=SRC["password"],
                         database=SRC["database"]) as src, src.cursor() as cur:
        for t in tables:
            cur.execute(f"SELECT COUNT(*) FROM `{t}`"); s = cur.fetchone()[0]
            d = int(s2_query(f"SELECT COUNT(*) AS n FROM `{S2['database']}`.`{t}`").n.iloc[0])
            rows.append({"table": t, "source_rows": s, "dest_rows": d, "match": "✔" if s == d else "✗"})
    return pd.DataFrame(rows)

display(reconcile())

In [21]:
# Health check you can schedule (e.g. as a SingleStore scheduled notebook job) and alert on.
def health_check(flow):
    src, dst = pairs_to_dict(flow.get_source()), pairs_to_dict(flow.get_destination())
    sched = flow.get_schedule()
    checks = {
        "source connection OK":      src.get("status") == "Y",
        "destination connection OK": dst.get("status") == "Y",
        "schedule active":           bool(sched.get("isScheduled")),
    }
    for k, ok in checks.items():
        print(("✔ " if ok else "✗ ") + k)
    return all(checks.values())

health_check(flow)

Pair this with the built-in pieces:
- **Monitor Flow on Helios with Alerts** — configure alerting on the instance in the Portal.
- **Reports** tab — per-table load volume and timing by day/month.
- **Suspend/Resume** and **Scale** — control cost and throughput.
- Schedule this notebook as a **Job** (Notebooks → Schedule) to run `health_check` + `reconcile` every N minutes.

### 7.2 · Where native Pipelines fit

| Need | Use |
|---|---|
| Replicate an operational DB (MySQL/Postgres/SQL Server/Oracle) with CDC | **Flow** (this notebook) |
| One-time migration of a large DB | **Flow + XL Ingest** |
| Event streams (Kafka, Confluent, Redpanda) | **Pipelines** `LOAD DATA KAFKA` |
| Files in object storage (S3, GCS, Azure Blob) | **Pipelines** `LOAD DATA S3` / `GCS` / `AZURE` |

> The older SQL-native "CDC-in" pipelines for MySQL (`CREATE LINK … AS MYSQL` + `CREATE TABLES AS INFER PIPELINE`) are **deprecated** in the SingleStore docs. Position Flow as the CDC path going forward.

The cell below shows the shape of a native Kafka and S3 pipeline for contrast. It is printed by default; set `RUN_PIPELINE_SQL = True` and fill in real broker/bucket values to run it.

In [22]:
RUN_PIPELINE_SQL = False

PIPELINE_SQL = f"""
CREATE TABLE IF NOT EXISTS clickstream (
    event_id BIGINT, customer_id BIGINT, event_type VARCHAR(32), ts DATETIME(6),
    SORT KEY (ts), SHARD KEY (customer_id)
);

CREATE OR REPLACE PIPELINE clickstream_kafka AS
  LOAD DATA KAFKA '<broker-host>:9092/clickstream'
  INTO TABLE clickstream
  FORMAT JSON (event_id <- id, customer_id <- customer_id, event_type <- type, ts <- ts);

CREATE OR REPLACE PIPELINE orders_archive_s3 AS
  LOAD DATA S3 '<bucket>/orders/*.csv'
  CONFIG '{{"region": "us-east-1"}}'
  CREDENTIALS '{{"aws_access_key_id": "<key>", "aws_secret_access_key": "<secret>"}}'
  INTO TABLE orders
  FIELDS TERMINATED BY ',' IGNORE 1 LINES;

START PIPELINE clickstream_kafka;
"""

if RUN_PIPELINE_SQL and not DRY_RUN:
    with s2_conn() as c, c.cursor() as cur:
        for stmt in [s for s in PIPELINE_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
    display(s2_query("SELECT pipeline_name, state FROM information_schema.pipelines"))
else:
    print(PIPELINE_SQL)

---
## Wrap-up

**What we showed**
- **Console:** Load Data → source tile → destination → dashboard checklist → start, with no infrastructure to run.
- **Connectors:** one `type` per capture mechanism — binlog, WAL, SQL Server CDC / Change Tracking, LogMiner, JDBC full extracts.
- **APIs:** a ~150-line client covering connections, tables, schedule, operations and XL Ingest.
- **Automation:** `provision_pipeline()` turns a pipeline into code you can version, review and re-run.
- **CDC:** insert/update/delete on the source show up in SingleStore within one schedule interval, verified from the destination side.

**Suggested next steps for the audience**
1. Point the notebook at a real source; set `DRY_RUN = False`.
2. Move `provision_pipeline` into CI with secrets from your vault.
3. Schedule `health_check` + `reconcile` as a notebook job and wire alerts.

**References** (SingleStore docs)
- Load Data with SingleStore Flow on Helios — https://docs.singlestore.com/cloud/load-data/load-data-with-singlestore-flow-on-helios/
- Use Flow on Helios — https://docs.singlestore.com/cloud/load-data/load-data-with-singlestore-flow-on-helios/use-flow-on-helios/
- Source Database Setup — https://docs.singlestore.com/cloud/load-data/load-data-with-singlestore-flow-on-helios/singlestore-ingest/source-database-setup/
- Flow on Helios API (auth, Ingest, XL Ingest) — https://docs.singlestore.com/cloud/load-data/load-data-with-singlestore-flow-on-helios/flow-on-helios-api/
- Flow release notes — https://docs.singlestore.com/cloud/load-data/load-data-with-singlestore-flow/singlestore-flow-release-notes/

In [23]:
# Final audit trail of every Flow API call made in this session
display(flow.audit())

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>